# Lenormand B4-E2 — sklearn 1.7.2 Reproducibility Audit

这个 notebook **不训练模型、不加载 27B、不检查 CUDA kernels**。CPU runtime 即可。

它只做四件事：

1. 将 NumPy/SciPy/Pandas/scikit-learn/joblib 锁回冻结模型的环境；
2. 对 Fold1/2 输入、冻结模型和配置计算 SHA256；
3. 在 sklearn 1.7.2 下重新应用冻结 B4-E2 并计算官方 one-to-one Phrase-F1；
4. 与 sklearn 1.9.0 那次结果逐值对比并打包轻量报告。

预计 2–6 分钟。不会改写 Task1 adapters、candidate margins 或 Fold1/2 评分缓存。

In [ ]:
#@title 0. 锁定科学计算栈（首次会自动重启；重连后再运行本格）
import importlib.metadata as metadata
import os, subprocess, sys

PINNED = {
    'numpy': '2.3.4',
    'scipy': '1.16.3',
    'pandas': '2.3.3',
    'scikit-learn': '1.7.2',
    'joblib': '1.5.2',
}
installed = {}
for package in PINNED:
    try:
        installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed[package] = None

if installed != PINNED:
    print('Repairing binary stack:', installed, '->', PINNED)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        '--force-reinstall', '--only-binary=:all:',
        'numpy==2.3.4', 'scipy==1.16.3', 'pandas==2.3.3',
        'scikit-learn==1.7.2', 'joblib==1.5.2',
    ])
    print('安装完成，正在重启 kernel。重连后再次运行 Cell 0。')
    os.kill(os.getpid(), 9)
else:
    import numpy as np, pandas as pd, scipy, sklearn, joblib
    print({'python': sys.version.split()[0], **installed, 'binary_stack': 'PASS'})
    assert sklearn.__version__ == '1.7.2'

In [ ]:
#@title 1. Drive、模块与只读输入路径
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import hashlib, importlib, json, shutil, sys
import numpy as np
import pandas as pd
import scipy, sklearn, joblib

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
FOLD_REFERENCE = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'
TASK1_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_OUTER_CONFIRM' / 'Q38_FULL64'
B4E2_ROOT = ROOT / 'results' / 'B4E2_CANDIDATE_META_FOLD0'
OUTPUT_ROOT = (
    ROOT / 'results' / 'B4_TASK1_Q38_FULL64_OUTER_CONFIRM' /
    'B4E2_OUTER_CONFIRMATION_SK172'
)
META_MODEL_PATH = B4E2_ROOT / 'B4E2_FOLD0_META_CALIBRATOR.joblib'
META_CONFIG_PATH = B4E2_ROOT / 'B4E2_FROZEN_CONFIG.json'
FACTOR_MACRO_F1 = 0.691954
RANK8_REFERENCE = 0.7615

MODULES = {
    'b1_experiments.py': None,
    'b4p_anchor_verifier.py': None,
    'qwen38_dual_task_experiments.py': 'Q38_RUNTIME_REVISION = \"2026-08-24.official-evidence-scorer-v3\"',
    'b4e_evidence_set.py': 'B4E_RUNTIME_REVISION = \"2026-08-24.official-one-to-one-event-set-v1\"',
    'b4e_candidate_meta.py': 'save_candidate_audits: bool = True',
}
stale = []
for name, marker in MODULES.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('请上传并覆盖：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded:
            raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

FOLD_ARTIFACTS = {}
for fold in (1, 2):
    eval_dir = TASK1_ROOT / f'fold_{fold}' / 'EVALUATION'
    FOLD_ARTIFACTS[fold] = (
        eval_dir / 'evidence_candidate_audit.csv',
        eval_dir / 'validation_predictions.csv',
    )
for path in [TRAIN_PATH, FOLD_REFERENCE, META_MODEL_PATH, META_CONFIG_PATH]:
    assert path.exists(), path
for pair in FOLD_ARTIFACTS.values():
    for path in pair:
        assert path.exists(), path
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
print({
    'sklearn': sklearn.__version__,
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'pandas': pd.__version__,
    'joblib': joblib.__version__,
    'output': str(OUTPUT_ROOT),
})
assert sklearn.__version__ == '1.7.2'

In [ ]:
#@title 2. 导入 scorer/meta 模块；不导入训练 harness
import b1_experiments as b1
import qwen38_dual_task_experiments as q38
import b4e_evidence_set as b4e
import b4e_candidate_meta as b4e2
importlib.reload(b1); importlib.reload(q38); importlib.reload(b4e); importlib.reload(b4e2)

assert q38.Q38_RUNTIME_REVISION == '2026-08-24.official-evidence-scorer-v3'
assert b4e.B4E_RUNTIME_REVISION == '2026-08-24.official-one-to-one-event-set-v1'
assert b4e2.B4E2_RUNTIME_REVISION == '2026-08-24.candidate-meta-event-decoder-v2'
assert hasattr(b4e2, 'confirm_frozen_meta_decoder')
print('Inference/scorer modules: PASS; no 27B model loaded.')

In [ ]:
#@title 3. 输入哈希、Fold 覆盖与冻结配置审计
def sha256(path, chunk_size=2**20):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

bundle = b1.load_training_data(ROOT, TRAIN_PATH)
reference = np.load(FOLD_REFERENCE, allow_pickle=True)
assert reference['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = reference['folds'].astype(int)

frozen_config = json.loads(META_CONFIG_PATH.read_text(encoding='utf-8'))
expected_config = {
    'logistic_c': 0.1,
    'probability_threshold': 0.4,
    'event_gap_chars': 40,
    'top_k_indicator': 0,
    'top_k_ideation': 2,
    'top_k_behavior': 3,
    'top_k_attempt': 2,
    'max_iter': 1000,
    'seed': 20260824,
}
assert frozen_config == expected_config

hash_records = []
for role, path in [
    ('meta_model', META_MODEL_PATH),
    ('meta_config', META_CONFIG_PATH),
    *[(f'fold_{fold}_candidate_audit', pair[0]) for fold, pair in FOLD_ARTIFACTS.items()],
    *[(f'fold_{fold}_validation_predictions', pair[1]) for fold, pair in FOLD_ARTIFACTS.items()],
]:
    hash_records.append({
        'role': role, 'path': str(path), 'bytes': path.stat().st_size,
        'sha256': sha256(path),
    })
hash_table = pd.DataFrame(hash_records)
hash_table.to_csv(OUTPUT_ROOT / 'B4E2_REPRO_INPUT_HASHES.csv', index=False)
display(hash_table)
print('Frozen config and input hashes: PASS')

In [ ]:
#@title 4. sklearn 1.7.2 下重新应用冻结 B4-E2（CPU 约 1–4 分钟）
canonical_decision = b4e2.confirm_frozen_meta_decoder(
    fold_artifacts=FOLD_ARTIFACTS,
    bundle=bundle,
    folds=folds,
    model_path=META_MODEL_PATH,
    config_path=META_CONFIG_PATH,
    output_dir=OUTPUT_ROOT,
    factor_macro_f1=FACTOR_MACRO_F1,
    rank8_reference=RANK8_REFERENCE,
    save_candidate_audits=False,
)
print(json.dumps(canonical_decision, ensure_ascii=False, indent=2))

In [ ]:
#@title 5. 与 sklearn 1.9.0 结果逐值对照
PRIOR_SK19 = {
    'fold_1_meta_f1': 0.7730334122599316,
    'fold_2_meta_f1': 0.7487073996265172,
    'pooled_meta_f1': 0.7608592164249938,
    'pooled_delta': 0.012032335445398834,
    'composite': 0.7653193051513507,
}
fold_by_id = {item['outer_fold']: item for item in canonical_decision['outer_fold_results']}
pooled = canonical_decision['pooled']
CANONICAL_SK172 = {
    'fold_1_meta_f1': fold_by_id[1]['meta_evidence_f1'],
    'fold_2_meta_f1': fold_by_id[2]['meta_evidence_f1'],
    'pooled_meta_f1': pooled['meta_evidence']['f1'],
    'pooled_delta': pooled['delta_evidence_f1'],
    'composite': pooled['meta_composite_projection'],
}
comparison = pd.DataFrame([
    {
        'metric': key, 'sklearn_1_9_result': PRIOR_SK19[key],
        'canonical_sklearn_1_7_2': CANONICAL_SK172[key],
        'absolute_difference': abs(PRIOR_SK19[key] - CANONICAL_SK172[key]),
    }
    for key in PRIOR_SK19
])
comparison.to_csv(OUTPUT_ROOT / 'B4E2_SKLEARN_VERSION_COMPARISON.csv', index=False)
display(comparison)
VERSION_INVARIANT = bool(comparison.absolute_difference.max() <= 1e-12)
REPRO_CERTIFIED = bool(
    canonical_decision['accepted_for_final_test'] and VERSION_INVARIANT
)
repro_decision = {
    'canonical_environment': {
        'numpy': np.__version__, 'scipy': scipy.__version__,
        'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
    'version_invariant_to_prior_sk19_run': VERSION_INVARIANT,
    'maximum_absolute_metric_difference': float(comparison.absolute_difference.max()),
    'canonical_accepted_for_final_test': canonical_decision['accepted_for_final_test'],
    'reproducibility_certified': REPRO_CERTIFIED,
    'canonical_metrics': CANONICAL_SK172,
    'next_step': (
        'BUILD_FINAL_OOF_REFIT_AND_TEST_SUBMISSION' if REPRO_CERTIFIED
        else 'USE_SK172_CANONICAL_RESULT_AND_INVESTIGATE_VERSION_DELTA'
    ),
}
(OUTPUT_ROOT / 'B4E2_SK172_REPRO_DECISION.json').write_text(
    json.dumps(repro_decision, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(repro_decision, ensure_ascii=False, indent=2))

In [ ]:
#@title 6. 打包轻量复现报告
report_root = OUTPUT_ROOT / 'LIGHT_REPRO_REPORT'
report_root.mkdir(parents=True, exist_ok=True)
for name in [
    'B4E2_OUTER_CONFIRMATION_DECISION.json',
    'B4E2_OUTER_FOLD_RESULTS.csv',
    'B4E2_OUTER_PREDICTIONS.csv',
    'B4E2_REPRO_INPUT_HASHES.csv',
    'B4E2_SKLEARN_VERSION_COMPARISON.csv',
    'B4E2_SK172_REPRO_DECISION.json',
]:
    source = OUTPUT_ROOT / name
    if source.exists():
        shutil.copy2(source, report_root / name)
archive = shutil.make_archive('/content/B4E2_SK172_REPRO_AUDIT', 'zip', report_root)
print('Saved:', archive)
# files.download(archive)

## 判读

- `reproducibility_certified=true`：1.7.2 与先前 1.9.0 的最终指标逐值一致，确认结果可冻结。
- 若版本不一致：以本 notebook 的 sklearn 1.7.2 canonical 结果为准；27B 仍不需要重跑。
- 这个 notebook 不需要 GPU。运行期间没有任何 Qwen 权重下载或模型训练。